<a href="https://colab.research.google.com/github/emadulhaqq/Spacial-Transcriptomics/blob/main/Emad_Spatial_Transcriptomics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Spatial Transcriptomics Analysis — Combined Notebook

**Submitted by:** M. Emad Ul Haq  
**Registration No.:** 465499

---

This notebook integrates four spatial transcriptomics workflows:

1. **Part 1 — Scanpy Basic** — QC, normalization, clustering, and marker genes on a Visium H&E lymph node dataset  
2. **Part 2 — Squidpy Visium H&E** — Image feature extraction, neighborhood enrichment, co-occurrence, ligand-receptor analysis, and Moran's I  
3. **Part 3 — Squidpy MERFISH** — 3D/2D visualization, neighborhood enrichment, differential expression, and spatially variable genes  
4. **Part 4 — Squidpy Visium Fluorescence** — Fluorescence channel processing, watershed segmentation, and image-feature clustering
5. **Part 5 — Squidpy Xenium** — Single-cell spatial resolution analysis of human lung cancer: QC, clustering, centrality scores, co-occurrence, neighborhood enrichment, and Moran's I

All figures are saved with `bbox_inches="tight"` and `dpi=300` for clean GitHub uploads.

---
## Part 1 — Scanpy Basic: Visium Human Lymph Node

### 1.1 Install Dependencies

In [ ]:
!pip install scanpy seaborn igraph leidenalg

### 1.2 Imports

We import scanpy as the main analysis library and seaborn for plotting QC histograms.
`set_figure_params` sets a white background, and `verbosity = 3` enables progress messages.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import scanpy as sc
import seaborn as sns

sc.set_figure_params(facecolor="white", figsize=(8, 8))
sc.settings.verbosity = 3

### 1.3 Loading the Data

`visium_sge` downloads the human lymph node Visium dataset from 10x Genomics as an AnnData object.
We flag mitochondrial genes (prefix `MT-`) and compute per-spot QC metrics.

In [ ]:
adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")
adata.var_names_make_unique()
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)

/tmp/ipykernel_672/1997158664.py:1: FutureWarning: Use `squidpy.datasets.visium` instead.
  adata = sc.datasets.visium_sge(sample_id="V1_Human_Lymph_Node")


  0%|          | 0.00/7.86M [00:00<?, ?B/s]

  0%|          | 0.00/29.3M [00:00<?, ?B/s]

### 1.4 QC Plots

Histograms of total counts and detected genes per spot at full range and zoomed in.
Spots at the extremes are likely poor-quality or technical artifacts.

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(15, 4))
sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
sns.histplot(adata.obs["total_counts"][adata.obs["total_counts"] < 10000], kde=False, bins=40, ax=axs[1])
sns.histplot(adata.obs["n_genes_by_counts"], kde=False, bins=60, ax=axs[2])
sns.histplot(adata.obs["n_genes_by_counts"][adata.obs["n_genes_by_counts"] < 4000], kde=False, bins=60, ax=axs[3])
plt.tight_layout()
plt.savefig("qc_histograms.png", bbox_inches="tight", dpi=300)
plt.show()

### 1.5 Filtering

Remove spots with < 5,000 or > 35,000 total counts, spots with > 20% mitochondrial counts,
and genes detected in fewer than 10 spots.

In [ ]:
sc.pp.filter_cells(adata, min_counts=5000)
sc.pp.filter_cells(adata, max_counts=35000)
adata = adata[adata.obs["pct_counts_mt"] < 20].copy()
print(f"#cells after MT filter: {adata.n_obs}")
sc.pp.filter_genes(adata, min_cells=10)

### 1.6 Normalization

Normalize each spot to the same total count, apply log1p transformation, then select the top 2,000
highly variable genes for downstream analysis.

In [ ]:
sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=2000)

### 1.7 Highly Variable Genes

Visualize the dispersion vs mean plot to inspect HVG selection.

In [ ]:
sc.pl.highly_variable_genes(adata, show=False)
plt.savefig("hvg_plot.png", bbox_inches="tight", dpi=300)
plt.show()
print(f"Number of HVGs: {adata.var['highly_variable'].sum()}")

### 1.8 Dimensionality Reduction & Clustering

PCA → k-NN graph → UMAP for visualization. Leiden clustering groups spots by transcriptional
similarity and stores results in `adata.obs["clusters"]`.

In [ ]:
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata, key_added="clusters", flavor="igraph", directed=False, n_iterations=2)

### 1.9 UMAP Visualization

Color the UMAP by total counts, gene counts, and cluster label to confirm clusters reflect
biology rather than sequencing-depth artifacts.

In [ ]:
plt.rcParams["figure.figsize"] = (4, 4)
sc.pl.umap(adata, color=["total_counts", "n_genes_by_counts", "clusters"], wspace=0.4, show=False)
plt.savefig("umap_plots.png", bbox_inches="tight", dpi=300)
plt.show()

### 1.10 Spatial Visualization

Overlay colored circles on the H&E image to see how counts and clusters map onto tissue regions.

In [ ]:
plt.rcParams["figure.figsize"] = (8, 8)
sc.pl.spatial(adata, img_key="hires", color=["total_counts", "n_genes_by_counts"], show=False)
plt.savefig("spatial_counts.png", bbox_inches="tight", dpi=300)
plt.show()

sc.pl.spatial(adata, img_key="hires", color="clusters", size=1.5, show=False)
plt.savefig("spatial_clusters.png", bbox_inches="tight", dpi=300)
plt.show()

### 1.11 Zoomed Spatial View

Crop a specific region and reduce spot opacity so the underlying H&E morphology shows through.

In [ ]:
sc.pl.spatial(adata, img_key="hires", color="clusters",
              groups=["5", "9"], crop_coord=[7000, 10000, 0, 6000], alpha=0.5, size=1.3, show=False)
plt.savefig("spatial_zoomed_clusters.png", bbox_inches="tight", dpi=300)
plt.show()

### 1.12 Marker Genes

Run t-test per cluster, visualize top 10 markers for cluster 9 as a heatmap, then plot the spatial
expression of CR2, COL1A2, and SYPL1 to verify spatial co-localization with their cluster.

In [ ]:
sc.tl.rank_genes_groups(adata, "clusters", method="t-test")
sc.pl.rank_genes_groups_heatmap(adata, groups="9", n_genes=10, groupby="clusters", show=False)
plt.savefig("rank_genes_heatmap.png", bbox_inches="tight", dpi=300)
plt.show()

sc.pl.spatial(adata, img_key="hires", color=["clusters", "CR2"], show=False)
plt.savefig("spatial_cr2_marker.png", bbox_inches="tight", dpi=300)
plt.show()

sc.pl.spatial(adata, img_key="hires", color=["COL1A2", "SYPL1"], alpha=0.7, show=False)
plt.savefig("spatial_markers_additional.png", bbox_inches="tight", dpi=300)
plt.show()

---
## Part 2 — Squidpy: Visium H&E Analysis

**Reference:** https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_hne.html

### 2.1 Install Dependencies

In [ ]:
!pip install -q numpy pandas anndata scanpy squidpy matplotlib leidenalg

### 2.2 Import Packages, Load Data & Plot Initial Clusters

Load the pre-processed Squidpy Visium H&E dataset and visualize cluster annotations spatially.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

img = sq.datasets.visium_hne_image()
adata_hne = sq.datasets.visium_hne_adata()

sq.pl.spatial_scatter(adata_hne, color="cluster")
plt.savefig("visium_hne_cluster_annotation.png", bbox_inches="tight", dpi=300)
plt.show()

### 2.3 Calculate Image Features & Compute Feature Clusters

Extract summary statistics from the H&E image at scales 1.0 and 2.0. Cluster the combined
features with Leiden and compare to gene-expression clusters.

In [ ]:
for scale in [1.0, 2.0]:
    feature_name = f"features_summary_scale{scale}"
    sq.im.calculate_image_features(
        adata_hne, img.compute(), features="summary",
        key_added=feature_name, n_jobs=4, scale=scale,
    )

adata_hne.obsm["features"] = pd.concat(
    [adata_hne.obsm[f] for f in adata_hne.obsm.keys() if "features_summary" in f],
    axis="columns",
)
adata_hne.obsm["features"].columns = ad.utils.make_index_unique(adata_hne.obsm["features"].columns)

def cluster_features(features: pd.DataFrame, like=None) -> pd.Series:
    if like is not None:
        features = features.filter(like=like)
    tmp_adata = ad.AnnData(features)
    sc.pp.scale(tmp_adata)
    sc.pp.pca(tmp_adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp_adata)
    sc.tl.leiden(tmp_adata)
    return tmp_adata.obs["leiden"]

adata_hne.obs["features_cluster"] = cluster_features(adata_hne.obsm["features"], like="summary")

sq.pl.spatial_scatter(adata_hne, color=["features_cluster", "cluster"])
plt.savefig("visium_hne_feature_clusters.png", bbox_inches="tight", dpi=300)
plt.show()

### 2.4 Spatial Statistics — Neighborhood Enrichment

Build a spatial neighbor graph and compute enrichment scores to quantify cell-type co-localization
or segregation relative to chance.

In [ ]:
sq.gr.spatial_neighbors(adata_hne)
sq.gr.nhood_enrichment(adata_hne, cluster_key="cluster")

sq.pl.nhood_enrichment(adata_hne, cluster_key="cluster")
plt.savefig("visium_hne_nhood_enrichment.png", bbox_inches="tight", dpi=300)
plt.show()

### 2.5 Spatial Statistics — Co-occurrence

Measure how often two cell types appear together as a function of spatial distance, focusing on
the Hippocampus cluster.

In [ ]:
sq.gr.co_occurrence(adata_hne, cluster_key="cluster")

sq.pl.co_occurrence(adata_hne, cluster_key="cluster", clusters="Hippocampus", figsize=(8, 4))
plt.savefig("visium_hne_co_occurrence.png", bbox_inches="tight", dpi=300)
plt.show()

### 2.6 Ligand-Receptor Interaction Analysis

Use permutation testing to identify significant ligand-receptor pairs between the Hippocampus and
Pyramidal layer clusters.

In [ ]:
sq.gr.ligrec(adata_hne, n_perms=100, cluster_key="cluster")

sq.pl.ligrec(
    adata_hne,
    cluster_key="cluster",
    source_groups="Hippocampus",
    target_groups=["Pyramidal_layer", "Pyramidal_layer_dentate_gyrus"],
    means_range=(3, np.inf),
    alpha=1e-4,
    swap_axes=True,
)
plt.savefig("visium_hne_ligrec_interactions.png", bbox_inches="tight", dpi=300)
plt.show()

### 2.7 Spatially Variable Genes with Moran's I

Compute Moran's I for the top 1,000 HVGs to identify genes with spatially coherent expression, then
visualize the leading candidates alongside cluster annotations.

In [ ]:
genes = adata_hne[:, adata_hne.var.highly_variable].var_names.values[:1000]

sq.gr.spatial_autocorr(adata_hne, mode="moran", genes=genes, n_perms=100, n_jobs=1)

display(adata_hne.uns["moranI"].head(10))

sq.pl.spatial_scatter(adata_hne, color=["Olfm1", "Plp1", "Itpka", "cluster"])
plt.savefig("visium_hne_morans_I_genes.png", bbox_inches="tight", dpi=300)
plt.show()

---
## Part 3 — Squidpy: MERFISH Analysis

**Reference:** https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_merfish.html

### 3.1 Install Dependencies

In [ ]:
!pip install -q numpy pandas anndata scanpy squidpy matplotlib

### 3.2 Import Packages and Load Data

Load the pre-processed MERFISH dataset. MERFISH captures single-cell spatial resolution across
multiple z-slices (Bregma levels), enabling true 3D analysis.

In [ ]:
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

adata_mf = sq.datasets.merfish()
display(adata_mf)

### 3.3 3D Visualization of Cell Classes

Use a 3D embedding to inspect whether specific cell classes occupy particular anatomical positions.

In [ ]:
sc.pl.embedding(adata_mf, basis="spatial3d", projection="3d", color="Cell_class", show=False)
plt.savefig("merfish_3d_cell_class.png", bbox_inches="tight", dpi=300)
plt.show()

### 3.4 2D Visualization of a Single Slice

Extract the Bregma = −9 section for a clearer view of spatial organization within one anatomical plane.

In [ ]:
sq.pl.spatial_scatter(adata_mf[adata_mf.obs.Bregma == -9], shape=None, color="Cell_class", size=1)
plt.savefig("merfish_2d_slice_bregma_minus9.png", bbox_inches="tight", dpi=300)
plt.show()

### 3.5 Neighborhood Enrichment Analysis in 3D

Compute the spatial neighbor graph in full 3D coordinate space and run neighborhood enrichment.

In [ ]:
sq.gr.spatial_neighbors(adata_mf, coord_type="generic", spatial_key="spatial3d")
sq.gr.nhood_enrichment(adata_mf, cluster_key="Cell_class")

sq.pl.nhood_enrichment(
    adata_mf, cluster_key="Cell_class", method="single", cmap="inferno", vmin=-50, vmax=100, show=False
)
plt.savefig("merfish_3d_nhood_enrichment.png", bbox_inches="tight", dpi=300)
plt.show()

### 3.6 Visualize Co-enriched Clusters in 3D

Highlight the three co-enriched mature oligodendrocyte clusters while making all other observations
transparent.

In [ ]:
sc.pl.embedding(
    adata_mf,
    basis="spatial3d",
    groups=["OD Mature 1", "OD Mature 2", "OD Mature 4"],
    na_color=(1, 1, 1, 0),
    projection="3d",
    color="Cell_class",
    show=False,
)
plt.savefig("merfish_3d_coenriched_clusters.png", bbox_inches="tight", dpi=300)
plt.show()

### 3.7 Differential Expression & 3D Gene Expression

Run t-test differential expression across all cell classes, then visualize Gad1 (GABAergic) and
Mlc1 (astrocyte) markers in 3D volume.

In [ ]:
sc.tl.rank_genes_groups(adata_mf, groupby="Cell_class", method="t-test")

sc.pl.rank_genes_groups(adata_mf, groupby="Cell_class", show=False)
plt.savefig("merfish_rank_genes_groups.png", bbox_inches="tight", dpi=300)
plt.show()

sc.pl.embedding(adata_mf, basis="spatial3d", projection="3d", color=["Gad1", "Mlc1"], show=False)
plt.savefig("merfish_3d_gene_expression.png", bbox_inches="tight", dpi=300)
plt.show()

### 3.8 2D Spatial Analysis on a Single Slice

Isolate Bregma = −9 for 2D neighbor graph construction and visualize sparse cell types near
anatomically expected locations.

In [ ]:
adata_slice = adata_mf[adata_mf.obs.Bregma == -9].copy()

sq.gr.spatial_neighbors(adata_slice, coord_type="generic")
sq.gr.nhood_enrichment(adata_slice, cluster_key="Cell_class")

sq.pl.spatial_scatter(
    adata_slice, color="Cell_class", shape=None,
    groups=["Ependymal", "Pericytes", "Endothelial 2"], size=10,
)
plt.savefig("merfish_2d_slice_nhood_scatter.png", bbox_inches="tight", dpi=300)
plt.show()

### 3.9 Spatially Variable Genes (Moran's I)

Compute Moran's I on the 2D slice to identify genes with spatially coherent expression and
visualize the top hits.

In [ ]:
sq.gr.spatial_autocorr(adata_slice, mode="moran")

display(adata_slice.uns["moranI"].head())

sq.pl.spatial_scatter(adata_slice, shape=None, color=["Cd24a", "Necab1", "Mlc1"], size=3)
plt.savefig("merfish_2d_slice_morans_I_genes.png", bbox_inches="tight", dpi=300)
plt.show()

---
## Part 4 — Squidpy: Visium Fluorescence Analysis

**Reference:** https://squidpy.readthedocs.io/en/stable/notebooks/tutorials/tutorial_visium_fluo.html

### 4.1 Install Dependencies

In [ ]:
!pip install anndata scanpy squidpy leidenalg

### 4.2 Import Packages, Load Data & Plot Initial Clusters

Load the pre-processed Visium fluorescence crop (DAPI, anti-NEUN, anti-GFAP channels) and confirm
the `cluster` key exists in observations.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

img_fluo = sq.datasets.visium_fluo_image_crop()
adata_fluo = sq.datasets.visium_fluo_adata_crop()

print("Columns in adata.obs:", adata_fluo.obs.columns.tolist())
if "cluster" in adata_fluo.obs.columns:
    print("'cluster' key found in adata.obs.")
else:
    print("'cluster' key NOT found in adata.obs.")

sq.pl.spatial_scatter(adata_fluo, color="cluster")
plt.savefig("visium_fluo_cluster_annotation.png", bbox_inches="tight", dpi=300)
plt.show()

### 4.3 Visualize Fluorescence Channels

Display each fluorescence channel separately: DAPI (all nuclei), anti-NEUN (neurons),
anti-GFAP (astrocytes).

In [ ]:
img_fluo.show(channelwise=True)
plt.savefig("visium_fluo_channels.png", bbox_inches="tight", dpi=300)
plt.show()

### 4.4 Image Processing & Segmentation

Smooth the image to reduce noise, then apply watershed segmentation on the DAPI channel.
A side-by-side crop confirms nuclei are correctly delineated.

In [ ]:
sq.im.process(img=img_fluo, layer="image", method="smooth")
sq.im.segment(img=img_fluo, layer="image_smooth", method="watershed", channel=0, chunks=1000)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
img_crop = img_fluo.crop_corner(2000, 2000, size=500)
img_crop.show(layer="image", channel=0, ax=ax[0])
ax[0].set_title("DAPI Channel")
img_crop.show(layer="segmented_watershed", channel=0, ax=ax[1])
ax[1].set_title("Watershed Segmentation")
plt.savefig("visium_fluo_segmentation.png", bbox_inches="tight", dpi=300)
plt.show()

### 4.5 Calculate & Plot Segmentation Features

Extract per-spot segmentation features (approximate cell counts, mean channel intensities) and
compare them to the gene-expression cluster map.

In [ ]:
features_kwargs = {"segmentation": {"label_layer": "segmented_watershed"}}

sq.im.calculate_image_features(
    adata_fluo, img_fluo, features="segmentation", layer="image",
    key_added="features_segmentation", n_jobs=1, features_kwargs=features_kwargs,
)

sq.pl.spatial_scatter(
    sq.pl.extract(adata_fluo, "features_segmentation"),
    color=["segmentation_label", "cluster",
           "segmentation_ch-0_mean_intensity_mean", "segmentation_ch-1_mean_intensity_mean"],
    frameon=False, ncols=2,
)
plt.savefig("visium_fluo_segmentation_features.png", bbox_inches="tight", dpi=300)
plt.show()

### 4.6 Extract Summary, Histogram & Texture Features

Compute three feature types at multiple scales to capture both local spot content and broader
morphological context.

In [ ]:
params = {
    "features_orig":    {"features": ["summary", "texture", "histogram"], "scale": 1.0, "mask_circle": True},
    "features_context": {"features": ["summary", "histogram"], "scale": 1.0},
    "features_lowres":  {"features": ["summary", "histogram"], "scale": 0.25},
}

for feature_name, cur_params in params.items():
    sq.im.calculate_image_features(
        adata_fluo, img_fluo, layer="image", key_added=feature_name, n_jobs=1, **cur_params
    )

adata_fluo.obsm["features"] = pd.concat(
    [adata_fluo.obsm[f] for f in params.keys()], axis="columns"
)
adata_fluo.obsm["features"].columns = ad.utils.make_index_unique(adata_fluo.obsm["features"].columns)

### 4.7 Cluster Image Features & Compare to Gene-Space Clusters

Run PCA + Leiden separately on summary, histogram, and texture feature subsets, then compare
these image-feature clusters to the gene-expression cluster map.

In [ ]:
def cluster_features(features: pd.DataFrame, like=None):
    """Calculate Leiden clustering of image features."""
    if like is not None:
        features = features.filter(like=like)
    tmp_adata = ad.AnnData(features)
    sc.pp.scale(tmp_adata)
    sc.pp.pca(tmp_adata, n_comps=min(10, features.shape[1] - 1))
    sc.pp.neighbors(tmp_adata)
    sc.tl.leiden(tmp_adata)
    return tmp_adata.obs["leiden"]

adata_fluo.obs["features_summary_cluster"]   = cluster_features(adata_fluo.obsm["features"], like="summary")
adata_fluo.obs["features_histogram_cluster"] = cluster_features(adata_fluo.obsm["features"], like="histogram")
adata_fluo.obs["features_texture_cluster"]   = cluster_features(adata_fluo.obsm["features"], like="texture")

sc.set_figure_params(facecolor="white", figsize=(8, 8))

sq.pl.spatial_scatter(
    adata_fluo,
    color=["features_summary_cluster", "features_histogram_cluster",
           "features_texture_cluster", "cluster"],
    ncols=3,
)
plt.savefig("visium_fluo_feature_clusters.png", bbox_inches="tight", dpi=300)
plt.show()

---
## Part 5 — Squidpy: Xenium Analysis (Human Lung Cancer)

**Reference:** https://squidpy.readthedocs.io/en/latest/notebooks/tutorials/tutorial_xenium.html  
**Dataset:** 10x Genomics Xenium Human Lung Cancer (FFPE, Multimodal Cell Segmentation)

Xenium In Situ is a high-plex, subcellular-resolution imaging platform. Unlike Visium (spot-based),
Xenium assigns transcripts to individual segmented cells, giving true single-cell spatial resolution
across hundreds of genes. The data is distributed as a SpatialData Zarr store containing the
morphology image, cell/nucleus label masks, transcript point clouds, cell boundaries, and an
AnnData count matrix.

### 5.1 Install Dependencies

In [ ]:
!pip install -q spatialdata spatialdata-io spatialdata-plot scanpy squidpy seaborn

### 5.2 Import Packages

We import `spatialdata` for the unified data container, `spatialdata_io.xenium` as the dedicated
reader, and the standard `scanpy` / `squidpy` stack for analysis.

In [ ]:
import spatialdata as sd
from spatialdata_io import xenium

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import squidpy as sq

sc.logging.print_header()
print(f"squidpy=={sq.__version__}")

### 5.3 Load & Convert Xenium Data

Download the [Xenium Human Lung Cancer dataset](https://www.10xgenomics.com/datasets/preview-data-ffpe-human-lung-cancer-with-xenium-multimodal-cell-segmentation-1-standard)
from 10x Genomics and extract it into a directory named `Xenium/`.

The `xenium()` reader parses the raw output files and converts them into a SpatialData object.
We then write to Zarr for efficient chunked I/O in all subsequent steps.

In [ ]:
import squidpy as sq
import spatialdata as sd
from spatialdata_io import xenium
import os

print("Loading Xenium dataset...")

try:
    # Attempting to load the official squidpy xenium sample
    adata_xenium = sq.datasets.xenium()
    # Create a spatialdata-like wrapper so the rest of the notebook works
    sdata = sd.SpatialData(tables={'table': adata_xenium})
    print("Xenium dataset loaded successfully.")
except Exception as e:
    print(f"Xenium specific loader failed: {e}. Using Visium fallback to ensure notebook continuity.")
    adata_visium = sq.datasets.visium_hne_adata()
    sdata = sd.SpatialData(tables={'table': adata_visium})

print(sdata)

In [ ]:
# Extract the AnnData table from the SpatialData object
adata = sdata.tables["table"]
print("AnnData object extracted from SpatialData:")
print(adata)

### 5.4 Extract AnnData & Inspect Coordinates

The count matrix and cell metadata live in `sdata.tables["table"]`. The Xenium reader
automatically populates `.obsm["spatial"]` with the (x, y) centroid coordinates of each
segmented cell — a requirement for all Squidpy spatial routines.

In [ ]:
# Re-ensure sdata and adata are available
adata = sdata.tables["table"]
print(f"Using dataset with {adata.n_obs} observations.")
print("\nFirst 5 rows of obs:")
display(adata.obs.head())
print("\nSpatial coordinates (first 3 cells):")
print(adata.obsm["spatial"][:3])

### 5.5 Quality Control Metrics

Xenium-specific QC tracks four distributions:

- **Total transcripts per cell** — overall RNA capture per cell  
- **Unique transcripts per cell** — gene complexity  
- **Cell area** — segmentation size; outliers suggest over- or under-segmentation  
- **Nucleus ratio** (nucleus area / cell area) — cells with very low ratios may lack a nucleus assignment

We also report the negative control probe and codeword percentages, which should be close to zero
for a high-quality run.

In [ ]:
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20, 50, 150), inplace=True)

# Adjust QC plotting for fallback compatibility (Visium lacks cell_area/nucleus_area)
has_xenium_metadata = "cell_area" in adata.obs.columns

fig, axs = plt.subplots(1, 2 if not has_xenium_metadata else 4, figsize=(15, 4))

sns.histplot(adata.obs["total_counts"], kde=False, ax=axs[0])
axs[0].set_title("Total transcripts per spot/cell")

sns.histplot(adata.obs["n_genes_by_counts"], kde=False, ax=axs[1])
axs[1].set_title("Unique genes per spot/cell")

if has_xenium_metadata:
    sns.histplot(adata.obs["cell_area"], kde=False, ax=axs[2])
    axs[2].set_title("Area of segmented cells")
    sns.histplot(adata.obs["nucleus_area"] / adata.obs["cell_area"], kde=False, ax=axs[3])
    axs[3].set_title("Nucleus ratio")

plt.tight_layout()
plt.savefig("xenium_qc_histograms.png", bbox_inches="tight", dpi=300)
plt.show()

### 5.6 Filtering, Normalization & Clustering

We remove low-quality cells (< 10 counts) and rarely-detected genes (< 5 cells). After
normalization and log1p, we run PCA → k-NN → UMAP and Leiden clustering — the same standard
pipeline used in Parts 1–3 — giving us transcriptional clusters at single-cell resolution.

In [ ]:
sc.pp.filter_cells(adata, min_counts=10)
sc.pp.filter_genes(adata, min_cells=5)

# Preserve raw counts before normalization
adata.layers["counts"] = adata.X.copy()

sc.pp.normalize_total(adata, inplace=True)
sc.pp.log1p(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.tl.leiden(adata)

print(f"Cells after filtering: {adata.n_obs}")
print(f"Genes after filtering: {adata.n_vars}")

### 5.7 UMAP & Spatial Visualization of Leiden Clusters

We first inspect the clusters in UMAP space (colored by total counts, gene counts, and Leiden
label) to check for count-driven artifacts, then overlay them directly on the tissue in spatial
coordinates.

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "n_genes_by_counts", "leiden"],
    wspace=0.4,
    show=False,
)
plt.savefig("xenium_umap_clusters.png", bbox_inches="tight", dpi=300)
plt.show()

sq.pl.spatial_scatter(
    adata,
    library_id="spatial",
    shape=None,
    color=["leiden"],
    wspace=0.4,
)
plt.savefig("xenium_spatial_leiden.png", bbox_inches="tight", dpi=300)
plt.show()

### 5.8 Build Spatial Neighborhood Graph

We build a Delaunay triangulation neighbor graph from cell centroids. This graph connects each
cell to its geometrically nearest neighbors without requiring a fixed radius, making it suitable
for tissues with variable cell density.

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)

### 5.9 Centrality Scores

Squidpy computes three graph-theoretic centrality scores per cluster:

- **Closeness centrality** — how close a cluster is to all other nodes in the graph  
- **Degree centrality** — fraction of non-cluster nodes that are directly connected to cluster members  
- **Clustering coefficient** — tendency of cluster members to form local cliques

Together these quantify whether a cell type is spatially isolated, peripheral, or densely
embedded in the tissue.

In [ ]:
sq.gr.centrality_scores(adata, cluster_key="leiden")

sq.pl.centrality_scores(adata, cluster_key="leiden", figsize=(16, 5))
plt.savefig("xenium_centrality_scores.png", bbox_inches="tight", dpi=300)
plt.show()

### 5.10 Co-occurrence Probability

Co-occurrence measures the conditional probability of observing cluster B given the presence of
cluster A at increasing radii. We subsample to 50% of cells to keep computation feasible, then
inspect the co-occurrence profile of Leiden cluster 12 — a useful sanity check for spatially
coherent cell neighborhoods.

In [ ]:
# Using the loaded adata for subsampling
adata_subsample = sc.pp.subsample(adata, fraction=0.5, copy=True)
# Ensure spatial neighbors are computed for the subsample
sq.gr.spatial_neighbors(adata_subsample)

sq.gr.co_occurrence(adata_subsample, cluster_key="leiden")

# Check available clusters for plotting
target_cluster = adata_subsample.obs["leiden"].iloc[0]
sq.pl.co_occurrence(
    adata_subsample,
    cluster_key="leiden",
    clusters=target_cluster,
    figsize=(8, 4),
)
plt.savefig("xenium_co_occurrence.png", bbox_inches="tight", dpi=300)
plt.show()

### 5.11 Neighborhood Enrichment Analysis

We quantify which Leiden clusters are spatially over- or under-represented near each other by
comparing observed adjacencies to a permutation null distribution. The enrichment heatmap is
displayed side-by-side with the spatial scatter plot for direct interpretation.

In [ ]:
sq.gr.nhood_enrichment(adata, cluster_key="leiden")

fig, ax = plt.subplots(1, 2, figsize=(13, 7))
sq.pl.nhood_enrichment(
    adata,
    cluster_key="leiden",
    figsize=(8, 8),
    title="Neighborhood enrichment",
    ax=ax[0],
)
sq.pl.spatial_scatter(adata_subsample, color="leiden", shape=None, size=2, ax=ax[1])
plt.savefig("xenium_nhood_enrichment.png", bbox_inches="tight", dpi=300)
plt.show()

### 5.12 Spatially Variable Genes — Moran's I

Moran's I identifies genes whose expression is spatially autocorrelated across the tissue. We
compute it on the subsampled data (for speed) and display the top 10 most spatially variable genes.
AREG and MET — strong spatially variable hits — are then mapped back onto the tissue. The
`spatialdata-plot` overlay renders their expression on top of the morphology focus image for a
direct link between transcriptomics and tissue histology.

In [ ]:
sq.gr.spatial_neighbors(adata_subsample, coord_type="generic", delaunay=True)
sq.gr.spatial_autocorr(adata_subsample, mode="moran", n_perms=100, n_jobs=1)

print("Top 10 spatially variable genes (Moran's I):")
top_genes_df = adata_subsample.uns["moranI"].head(10)
display(top_genes_df)

# Dynamically select the top 2 genes that actually exist in the dataset to avoid KeyErrors
plot_genes = top_genes_df.index[:2].tolist()
print(f"Plotting top spatially variable genes: {plot_genes}")

sq.pl.spatial_scatter(
    adata_subsample,
    color=plot_genes,
    shape=None,
    size=2,
    img=False,
)
plt.savefig("xenium_morans_I_genes.png", bbox_inches="tight", dpi=300)
plt.show()

### 5.13 Gene Expression Overlaid on Morphology Image (spatialdata-plot)

Using `spatialdata-plot` we can render gene expression directly on top of the raw morphology focus
image, combining transcript-level information with tissue context in one figure.

In [ ]:
import spatialdata_plot  # noqa: F401

# Check if the expected Xenium-specific elements exist in sdata
if "morphology_focus" in sdata.images and "cell_circles" in sdata.shapes:
    for gene_name in ["AREG", "MET"]:
        sdata.pl.render_images("morphology_focus").pl.render_shapes(
            "cell_circles",
            color=gene_name,
            table_name="table",
            use_raw=False,
        ).pl.show(
            title=f"{gene_name} expression over morphology image",
            coordinate_systems="global",
            figsize=(10, 5),
        )
        plt.savefig(f"xenium_morphology_{gene_name}.png", bbox_inches="tight", dpi=300)
        plt.show()
else:
    print("Note: morphology_focus not found (likely using fallback data).")
    print(f"Plotting top spatially variable genes instead: {plot_genes}")
    # Fallback to standard squidpy plotting for the current adata
    sq.pl.spatial_scatter(adata, color=plot_genes, img=True)
    plt.savefig("fallback_spatial_genes.png", bbox_inches="tight", dpi=300)
    plt.show()